# 🎙️ microWakeWord Training — Kokoro TTS + GPU Moderna (v2 Anti-FP)

Pipeline de entrenamiento para un modelo wake word compatible con **ESP32-S3** (via ESPHome/microWakeWord).  
Genera muestras de voz sintéticas con **Kokoro TTS** usando múltiples voces en español para mayor diversidad.

### ✨ Mejoras v2 (Anti-Falsos Positivos)
- **Simulación realista del micrófono INMP441** (HPF, ruido I2S, variación de ganancia)
- **40+ variantes fonéticas** de la wake word (acentos, puntuación, velocidades extremas)
- **Negativos en español** via Common Voice (crítico para wake words en español)
- **Entrenamiento en 2 fases** con penalización agresiva de falsos positivos
- **Voces adicionales** y variación de tono con PitchShift manual

### Requisitos del sistema
- Python 3.10 o 3.11
- GPU NVIDIA moderna (RTX 3000+ recomendada)
- CUDA 12.x + cuDNN 8.x
- ~30 GB de espacio libre en disco

### Flujo general
1. Instalar dependencias
2. Configurar GPU
3. Generar muestras con Kokoro TTS (variantes fonéticas + voces múltiples)
4. Descargar datasets negativos (incluyendo español)
5. Augmentar con simulación INMP441 y generar espectrogramas
6. Entrenar el modelo en 2 fases
7. Exportar `.tflite` para ESP32-S3

---

## 📦 Celda 1 — Instalación de dependencias
Ejecutar una sola vez. Reiniciar el kernel después si usas Jupyter clásico.

In [1]:
import platform, subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *args])

# --- microWakeWord ---
if platform.system() == 'Darwin':
    pip('git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version')

pip('git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f')

import os
if not os.path.exists('microWakeWord'):
    subprocess.check_call(['git', 'clone', 'https://github.com/kahrendt/microWakeWord'])

pip('-e', './microWakeWord')

# --- Kokoro TTS ---
pip('kokoro>=0.9.4', 'soundfile', 'numpy')
# Instala los componentes específicos que TF busca y que a veces Torch no incluye
pip('nvidia-cudnn-cu12==8.9.4.25', 'nvidia-cublas-cu12')
# --- Herramientas de audio y training ---
pip('datasets', 'scipy', 'tqdm', 'tensorboard', 'audiomentations', 'mmap_ninja')

print('✅ Instalación completa. Reinicia el kernel si es la primera vez.')


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


✅ Instalación completa. Reinicia el kernel si es la primera vez.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys
import subprocess

print("🧹 Limpiando...")
subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'tensorflow', 'tensorflow-cpu', 'tensorflow-gpu', 'keras'])

print("✨ Instalando TensorFlow 2.16.2 y el NumPy compatible...")
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tensorflow[and-cuda]==2.16.2', 'numpy<2.0.0'])

print("📦 Reinstalando microWakeWord para alinear dependencias...")
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', './microWakeWord'])

print("✅ ¡Listo! Por favor, REINICIA EL KERNEL ahora.")

🧹 Limpiando...
Found existing installation: tensorflow 2.16.2
Uninstalling tensorflow-2.16.2:
  Successfully uninstalled tensorflow-2.16.2


Found existing installation: keras 3.14.1
Uninstalling keras-3.14.1:
  Successfully uninstalled keras-3.14.1
✨ Instalando TensorFlow 2.16.2 y el NumPy compatible...
  Using cached tensorflow-2.16.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.2 kB)
  Using cached keras-3.14.1-py3-none-any.whl.metadata (6.3 kB)
  Using cached nvidia_cudnn_cu12-8.9.7.29-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nccl_cu12-2.19.3-py3-none-manylinux1_x86_64.whl.metadata (1.8 kB)
Using cached tensorflow-2.16.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (590.7 MB)
Using cached nvidia_cudnn_cu12-8.9.7.29-py3-none-manylinux1_x86_64.whl (704.7 MB)
Using cached nvidia_nccl_cu12-2.19.3-py3-none-manylinux1_x86_64.whl (166.0 MB)
Using cached keras-3.14.1-py3-none-any.whl (1.6 MB)
  Attempting uninstall: nvidia-nccl-cu12
    Found existing installation: nvidia-nccl-cu12 2.30.4
    Uninstalling nvidia-nccl-cu12-2.30.4:
      Successfully uninstalle


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


Obtaining file:///home/sesgaro/microwakeword/microWakeWord
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for microwakeword (pyproject.toml): started
  Building editable for microwakeword (pyproject.toml): finished with status 'done'
  Created wheel for microwakeword: filename=microwakeword-0.1.0-0.editable-py3-none-any.whl size=9942 sha256=c792d3a05222e41088ec4ef5366794666c0ec227cf0d5b2ea65422fd056314d9
  Stored in directory: /tmp/pip-ephem-wheel-cache-z3rn_obx/wheels/f9/b0/eb/37c9508bbff2551c0508a6a7b5b21721502bb06e8dfcf2ed


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## ⚡ Celda 2 — Configuración de GPU
Detecta y configura la GPU. Compatible con RTX 3000/4000 series, A100, etc.

In [1]:
import os, sys, glob

# 1. QUITAR EL SILENCIADOR (0 = Mostrar todos los errores de C++)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '0'  
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

# 2. Forzar rutas del entorno virtual
venv_base = sys.prefix
nvidia_path = os.path.join(venv_base, 'lib', f'python{sys.version_info.major}.{sys.version_info.minor}', 'site-packages', 'nvidia')
lib_dirs = glob.glob(os.path.join(nvidia_path, '*', 'lib'))
if os.path.exists('/usr/local/cuda/lib64'): 
    lib_dirs.append('/usr/local/cuda/lib64')
os.environ['LD_LIBRARY_PATH'] = ':'.join(lib_dirs) + ':' + os.environ.get('LD_LIBRARY_PATH', '')

# 3. Cargar TensorFlow y probar
import tensorflow as tf
tf.keras.backend.clear_session()
gpus = tf.config.list_physical_devices('GPU')

print(f'\n--- RESULTADOS ---')
print(f'TensorFlow versión: {tf.__version__}')
if gpus:
    print(f'🚀 ¡GPU DETECTADA!: {[g.name for g in gpus]}')
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print('❌ Sigue sin ver la GPU.')
    print('👆 REVISA ARRIBA: TensorFlow acaba de imprimir en pantalla qué archivo exacto le falta (probablemente un cudnn.so o libcublas.so).')

2026-05-13 10:29:15.696188: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-13 10:29:15.766095: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-13 10:29:15.766346: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-13 10:29:15.845789: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-13 10:29:16.848379: W tensorflow/compiler/tf


--- RESULTADOS ---
TensorFlow versión: 2.16.2
🚀 ¡GPU DETECTADA!: ['/physical_device:GPU:0']


2026-05-13 10:29:18.939021: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-13 10:29:19.058902: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-13 10:29:19.060924: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

## 🛠️ Celda 3 — Funciones auxiliares: Simulación INMP441 + variantes fonéticas

El **INMP441** es un micrófono MEMS I2S con características específicas que el modelo debe aprender:
- High-pass filter interno a ~60 Hz (elimina DC offset y ruido de muy baja frecuencia)
- Conversión I2S de 24-bit → 16-bit con ruido de cuantización
- Variación de ganancia por temperatura (±1 dB)
- Ligera resonancia en ~8-10 kHz característica de micrófonos MEMS

Las **variantes fonéticas** permiten que el modelo sea robusto a diferentes pronunciaciones,  
velocidades, énfasis y acentos regionales del español.

In [8]:
import numpy as np
from scipy.signal import butter, sosfilt, resample_poly

# ══════════════════════════════════════════════════════════════════════════════
#  🎤 SIMULACIÓN DEL MICRÓFONO INMP441
# ══════════════════════════════════════════════════════════════════════════════

def simulate_inmp441(samples: np.ndarray, sample_rate: int = 16000) -> np.ndarray:
    """
    Simula las características electro-acústicas del micrófono INMP441 + ESP32-S3 I2S.
    (Versión Segura: Ajustada a límites de Nyquist)
    """
    samples = samples.copy().astype(np.float64)

    # 1. High-pass filter interno del INMP441 (~60 Hz, 2do orden)
    sos_hpf = butter(2, 60.0, btype='high', fs=sample_rate, output='sos')
    samples = sosfilt(sos_hpf, samples)

    # 2. Ruido térmico del JFET (~-110 dBFS)
    thermal_noise_level = 10 ** (-110 / 20.0)
    samples += np.random.normal(0, thermal_noise_level, len(samples))

    # 3. Ruido de cuantización 24→16 bit (I2S shifting)
    quant_noise_level = 0.5 / (2 ** 15)
    samples += np.random.uniform(-quant_noise_level, quant_noise_level, len(samples))

    # 4. Variación de ganancia por temperatura (±1.5 dB aleatorio)
    gain_db = np.random.uniform(-1.5, 1.5)
    samples *= 10 ** (gain_db / 20.0)

    # 5. Resonancia MEMS sutil en altas frecuencias
    nyquist = sample_rate / 2.0
    
    # 🛡️ PARCHE DSP: Asegurar que la resonancia no rompa Nyquist
    max_res = min(10000, nyquist * 0.95) # Máximo 95% de Nyquist
    min_res = min(7500, nyquist * 0.80)
    
    if max_res > min_res:
        resonance_freq = np.random.uniform(min_res, max_res)
        resonance_gain = np.random.uniform(0.5, 2.0)  # ±0.5-2 dB
        Q = 3.0  # Factor de calidad
        w0 = resonance_freq / nyquist
        
        # Filtro peaking simple
        alpha = np.sin(np.pi * w0) / (2 * Q)
        A = 10 ** (resonance_gain / 40.0)
        
        b_peak = np.array([1 + alpha * A, -2 * np.cos(np.pi * w0), 1 - alpha * A])
        a_peak = np.array([1 + alpha / A, -2 * np.cos(np.pi * w0), 1 - alpha / A])
        
        from scipy.signal import lfilter
        samples = lfilter(b_peak, a_peak, samples)

    return np.clip(samples, -1.0, 1.0).astype(np.float32)


# ══════════════════════════════════════════════════════════════════════════════
#  📝 VARIANTES FONÉTICAS DE LA WAKE WORD
# ══════════════════════════════════════════════════════════════════════════════

def get_wake_word_variants(base_word: str) -> list:
    """
    Genera variantes fonéticas, ortográficas y prosódicas de la wake word.

    Kokoro TTS interpreta diferente:
      - Puntuación: genera pausas, énfasis, entonación interrogativa/exclamativa
      - Ortografía alternativa: cambia la pronunciación (y→i, j→h, etc.)
      - Mayúsculas: algunos engines generan más énfasis
      - Tildes y diacríticos: corrigen el acento prosódico
      - Separación de sílabas con guion: alarga vocales, aclara articulación

    Devuelve lista de (texto, descripcion, peso_relativo)
    """
    # Para 'jey ardo' — ajusta según tu wake word
    variants = [
        # ── Forma canónica ─────────────────────────────────────────────────────
        (base_word,                   'base',                       1.0),

        # ── Variantes ortográficas (misma pronunciación objetivo) ──────────────
        ('hey ardo',                  'ortografia_hey',             0.9),
        ('ei ardo',                   'vocal_ei',                   0.6),
        ('ey ardo',                   'sin_j',                      0.7),
        ('jei ardo',                  'ortografia_jei',             0.8),
        ('je ardo',                   'sin_y_final',                0.5),

        # ── Con puntuación — genera pausas y entonación variable ───────────────
        ('jey, ardo',                 'coma_pausa',                 0.8),
        ('jey! ardo',                 'exclamacion_enfasis',        0.7),
        ('¡jey ardo!',               'exclamacion_doble',          0.7),
        ('jey ardo.',                 'punto_final',                0.6),
        ('jey... ardo',               'puntos_pausa_larga',         0.5),
        ('jey ardo,',                 'coma_al_final',              0.4),
        ('¡jey! ardo',               'exclamacion_solo_jey',       0.6),
        ('jey ardo!',                 'exclamacion_final',          0.6),

        # ── Con tildes — corrige acento prosódico en Kokoro ───────────────────
        ('jéy ardo',                  'tilde_jey',                  0.7),
        ('jey ardó',                  'tilde_ardo',                 0.7),
        ('jéy ardó',                  'tilde_ambas',                0.5),
        ('héy ardo',                  'tilde_hey',                  0.6),

        # ── Sílabas separadas — Kokoro alarga vocales y aclara articulación ──
        ('je-y ar-do',                'silabas_guion',              0.5),
        ('jey ar-do',                 'silaba_ardo',                0.5),

        # ── Mayúsculas — énfasis en Kokoro ────────────────────────────────────
        ('JEY ardo',                  'mayus_jey',                  0.4),
        ('jey ARDO',                  'mayus_ardo',                 0.4),
        ('JEY ARDO',                  'mayus_completo',             0.4),
        ('Jey Ardo',                  'capitalizado',               0.6),

        # ── Variantes rápidas (pronunciación natural apresurada) ──────────────
        ('jeyardo',                   'fusion_sin_espacio',         0.6),
        ('hey-ardo',                  'guion_fusion',               0.5),

        # ── Variantes con contexto — entrena el modelo a detectar en oración ──
        ('oye jey ardo',              'con_oye',                    0.4),
        ('hola jey ardo',             'con_hola',                   0.4),
        ('eh jey ardo',               'con_eh',                     0.3),
        ('bueno jey ardo',            'con_bueno',                  0.3),

        # ── Variantes con diferentes velocidades implícitas ────────────────────
        # (además se varía speed= en el pipeline)
        ('jey... ardo.',              'lento_pausado',              0.5),
        ('¡jey ardo, escucha!',      'con_contexto_orden',         0.3),

        # ── Acentos regionales (ortografía diferente para guiar Kokoro) ────────
        # México / Caribe — yeísmo fuerte
        ('yey ardo',                  'acento_mexico_yey',          0.5),
        # Argentina — 'sh' para 'll' y 'y'
        ('shey ardo',                 'acento_argentina',           0.4),
        # España — 'j' más fricativa
        ('jéi ardo',                  'acento_espana',              0.4),
        # Caribe — vocal final abierta
        ('jey ardoh',                 'acento_caribe',              0.3),
        # Andino — muy articulado
        ('jey ar-do',                 'acento_andino',              0.4),
    ]
    return variants


# Test rápido
variants = get_wake_word_variants('jey ardo')
print(f'📝 Total de variantes fonéticas: {len(variants)}')
print('\nVariantes definidas:')
for text, desc, weight in variants:
    print(f'  [{weight:.1f}] "{text}" ({desc})')

📝 Total de variantes fonéticas: 37

Variantes definidas:
  [1.0] "jey ardo" (base)
  [0.9] "hey ardo" (ortografia_hey)
  [0.6] "ei ardo" (vocal_ei)
  [0.7] "ey ardo" (sin_j)
  [0.8] "jei ardo" (ortografia_jei)
  [0.5] "je ardo" (sin_y_final)
  [0.8] "jey, ardo" (coma_pausa)
  [0.7] "jey! ardo" (exclamacion_enfasis)
  [0.7] "¡jey ardo!" (exclamacion_doble)
  [0.6] "jey ardo." (punto_final)
  [0.5] "jey... ardo" (puntos_pausa_larga)
  [0.4] "jey ardo," (coma_al_final)
  [0.6] "¡jey! ardo" (exclamacion_solo_jey)
  [0.6] "jey ardo!" (exclamacion_final)
  [0.7] "jéy ardo" (tilde_jey)
  [0.7] "jey ardó" (tilde_ardo)
  [0.5] "jéy ardó" (tilde_ambas)
  [0.6] "héy ardo" (tilde_hey)
  [0.5] "je-y ar-do" (silabas_guion)
  [0.5] "jey ar-do" (silaba_ardo)
  [0.4] "JEY ardo" (mayus_jey)
  [0.4] "jey ARDO" (mayus_ardo)
  [0.4] "JEY ARDO" (mayus_completo)
  [0.6] "Jey Ardo" (capitalizado)
  [0.6] "jeyardo" (fusion_sin_espacio)
  [0.5] "hey-ardo" (guion_fusion)
  [0.4] "oye jey ardo" (con_oye)
  [0.4] 

In [2]:
import sys
import subprocess

print("🩹 Restaurando el motor NCCL de NVIDIA...")
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--force-reinstall', 'nvidia-nccl-cu12'])

print("✅ Librería restaurada con éxito.")

🩹 Restaurando el motor NCCL de NVIDIA...
  Using cached nvidia_nccl_cu12-2.30.4-py3-none-manylinux_2_18_x86_64.whl.metadata (2.1 kB)
Using cached nvidia_nccl_cu12-2.30.4-py3-none-manylinux_2_18_x86_64.whl (300.2 MB)
  Attempting uninstall: nvidia-nccl-cu12
    Found existing installation: nvidia-nccl-cu12 2.19.3
    Uninstalling nvidia-nccl-cu12-2.19.3:
      Successfully uninstalled nvidia-nccl-cu12-2.19.3
✅ Librería restaurada con éxito.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## 🗣️ Celda 4 — Generación de muestras con Kokoro TTS

Genera ~1500+ muestras usando **variantes fonéticas + múltiples voces** de Kokoro.  
Cada muestra pasa por la **simulación INMP441** para que el modelo aprenda el color real del micrófono.

**Voces disponibles en español (Kokoro):**
- `ef_dora` — femenina, acento neutro latinoamericano
- `em_alex` — masculina, acento neutro latinoamericano  
- `em_santa` — masculina, tono más grave
- `ef_bella` — femenina, tono más suave

In [11]:

import os

# =======================================================
# 🛡️ PARCHE DEFINITIVO ANTI-COLAPSO C++
# Apagamos los compiladores "al vuelo" de PyTorch.
# ¡ESTO DEBE IR ESTRICTAMENTE ANTES DE IMPORTAR TORCH!
# =======================================================
os.environ['PYTORCH_JIT'] = '0'
os.environ['PYTORCH_TENSOREXPR'] = '0'

import torch
import random
import numpy as np
import soundfile as sf
from tqdm import tqdm
from kokoro import KPipeline
from scipy.signal import resample_poly


# ══════════════════════════════════════════════════════════════════════════════
#  ⚙️  CONFIGURACIÓN — Ajusta estos valores
# ══════════════════════════════════════════════════════════════════════════════
TARGET_WORD    = 'jey ardo'        # Tu wake word base
OUTPUT_DIR     = 'generated_samples'
SAMPLES_TOTAL  = 1500              # Total de muestras a generar
KOKORO_DEVICE  = 'cpu'            # 'cuda' para GPU, 'cpu' para fallback
APPLY_INMP441  = True              # Aplicar simulación INMP441 en cada muestra

# Voces y su peso relativo
VOICE_CONFIG = [
    # (nombre_voz,   peso,  rango_velocidad)
    ('em_alex',      0.35,  (0.80, 1.20)),   # 35% de las muestras
    ('ef_dora',      0.35,  (0.80, 1.20)),   # 35% de las muestras
    ('em_santa',     0.30,  (0.75, 1.10)),   # 30% de las muestras
]
# ══════════════════════════════════════════════════════════════════════════════

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Obtener variantes fonéticas (asume que la función ya se ejecutó en la celda anterior)
variants = get_wake_word_variants(TARGET_WORD)
variant_texts   = [v[0] for v in variants]
variant_weights = [v[2] for v in variants]
total_weight_v  = sum(variant_weights)
variant_probs   = [w / total_weight_v for w in variant_weights]

# Calcular cuántas muestras por voz
voice_counts = {}
total_weight_voices = sum(w for _, w, _ in VOICE_CONFIG)
remaining = SAMPLES_TOTAL
for i, (voice, weight, _) in enumerate(VOICE_CONFIG):
    if i < len(VOICE_CONFIG) - 1:
        count = round(SAMPLES_TOTAL * weight / total_weight_voices)
        remaining -= count
    else:
        count = remaining
    voice_counts[voice] = count

print(f"🎙️  Wake word base: '{TARGET_WORD}'")
print(f"📝  Variantes fonéticas disponibles: {len(variants)}")
print(f"🔊  Simulación INMP441: {'✅ Activada' if APPLY_INMP441 else '❌ Desactivada'}")
print(f"\n📊  Distribución de muestras:")
for voice, _, _ in VOICE_CONFIG:
    print(f"    • {voice}: {voice_counts[voice]} muestras")
print(f"    TOTAL: {SAMPLES_TOTAL} muestras\n")

sample_idx = 0
errors = 0

for voice_name, weight, speed_range in VOICE_CONFIG:
    n_samples = voice_counts[voice_name]
    print(f"⏳ Generando {n_samples} muestras con voz '{voice_name}'...")

    try:
        pipeline = KPipeline(lang_code='e', device=KOKORO_DEVICE)
    except Exception as e:
        print(f"  ⚠️  No se pudo cargar en {KOKORO_DEVICE}, usando CPU. Error: {e}")
        pipeline = KPipeline(lang_code='e', device='cpu')

    with tqdm(total=n_samples, desc=f"  {voice_name}") as pbar:
        for i in range(n_samples):
            # Seleccionar variante fonética según distribución de pesos
            text = np.random.choice(variant_texts, p=variant_probs)
            # Velocidad aleatoria en el rango de la voz
            speed = round(random.uniform(*speed_range), 3)
            out_path = os.path.join(OUTPUT_DIR, f"{sample_idx:05d}.wav")

            try:
                # 🛡️ Envolvemos la inferencia para asegurar que no se guarde historial de gradientes
                with torch.no_grad():
                    generator = pipeline(text, voice=voice_name, speed=speed)
                    chunks = [audio for _, _, audio in generator]

                if chunks:
                    audio_full = np.concatenate(chunks)
                    # Kokoro genera a 24kHz → resamplear a 16kHz
                    audio_16k = resample_poly(audio_full, up=2, down=3)

                    # Normalizar antes de la simulación
                    max_val = np.max(np.abs(audio_16k))
                    if max_val > 0:
                        audio_16k = audio_16k / max_val

                    # Aplicar ganancia aleatoria ANTES de simular el micrófono
                    gain = round(random.uniform(0.15, 1.0), 3)
                    audio_16k = audio_16k * gain

                    # Simulación del micrófono INMP441
                    if APPLY_INMP441:
                        audio_16k = simulate_inmp441(audio_16k, sample_rate=16000)

                    audio_16k = np.clip(audio_16k, -1.0, 1.0)
                    audio_16k_int = (audio_16k * 32767).astype(np.int16)
                    sf.write(out_path, audio_16k_int, 16000)
                    sample_idx += 1
                else:
                    errors += 1
                    print(f"  ⚠️  Kokoro no generó audio para: '{text}'")

            except Exception as e:
                errors += 1
                if errors <= 5:  # Solo mostrar primeros 5 errores
                    print(f"  ❌ Error en muestra {sample_idx} ('{text}'): {e}")

            pbar.update(1)

    # Limpiar RAM/VRAM
    del pipeline
    torch.cuda.empty_cache()
    print()

print(f"✅ Generación completa: {sample_idx} muestras guardadas en '{OUTPUT_DIR}/'")
if errors > 0:
    print(f"⚠️  {errors} errores durante la generación (variantes no soportadas por Kokoro)")

🎙️  Wake word base: 'jey ardo'
📝  Variantes fonéticas disponibles: 37
🔊  Simulación INMP441: ✅ Activada

📊  Distribución de muestras:
    • em_alex: 525 muestras
    • ef_dora: 525 muestras
    • em_santa: 450 muestras
    TOTAL: 1500 muestras

⏳ Generando 525 muestras con voz 'em_alex'...


  em_alex: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 525/525 [01:58<00:00,  4.42it/s]



⏳ Generando 525 muestras con voz 'ef_dora'...


  ef_dora: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 525/525 [01:57<00:00,  4.45it/s]



⏳ Generando 450 muestras con voz 'em_santa'...


  em_santa: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 450/450 [01:53<00:00,  3.96it/s]


✅ Generación completa: 1500 muestras guardadas en 'generated_samples/'


## 🔊 Celda 5 — Verificar muestras generadas
Reproduce muestras al azar para verificar calidad y diversidad.

In [12]:
import random, glob
from IPython.display import Audio, display

samples = glob.glob(f'{OUTPUT_DIR}/*.wav')
print(f'📂 Total de muestras: {len(samples)}')

# Reproducir 3 muestras al azar
for i, sample in enumerate(random.sample(samples, min(3, len(samples)))):
    print(f'\n🎧 Muestra {i+1}: {sample}')
    display(Audio(sample, autoplay=(i == 0)))

📂 Total de muestras: 1500

🎧 Muestra 1: generated_samples/00321.wav



🎧 Muestra 2: generated_samples/01250.wav



🎧 Muestra 3: generated_samples/00699.wav


## 📥 Celda 6 — Descargar datasets negativos

Los negativos son **cruciales** para reducir falsos positivos.  
Esta versión incluye **Common Voice en español** — el dataset más importante para wake words en español,  
ya que el modelo aprende a ignorar fonemas y palabras similares a la wake word en el idioma correcto.

In [13]:
import os

# ── Datasets negativos pre-generados de microWakeWord ─────────────────────────
NEG_DIR = './negative_datasets'
if not os.path.exists(NEG_DIR):
    os.makedirs(NEG_DIR, exist_ok=True)
    link_root = 'https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/'
    filenames = ['dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip']
    for fname in filenames:
        zip_path = f'{NEG_DIR}/{fname}'
        print(f'⏳ Descargando {fname}...')
        os.system(f'wget -q --show-progress -O {zip_path} {link_root + fname}')
        os.system(f'unzip -q {zip_path} -d {NEG_DIR}')
        os.remove(zip_path)
    print('✅ Datasets negativos base descargados.')
else:
    print('✅ Datasets negativos base ya existen. Saltando descarga.')

✅ Datasets negativos base ya existen. Saltando descarga.


In [14]:
import datasets as hf_datasets
import scipy.io.wavfile
import numpy as np
import os
from tqdm import tqdm

# ══════════════════════════════════════════════════════════════════════════════
#  🇪🇸 NEGATIVOS EN ESPAÑOL — Common Voice (Mozilla)
#  El más importante para reducir FP en wake words en español
# ══════════════════════════════════════════════════════════════════════════════
SPANISH_NEG_DIR = './negative_datasets/spanish_speech'
SPANISH_SAMPLES = 4000   # Más muestras = menos FP en español

if not os.path.exists(SPANISH_NEG_DIR) or len(os.listdir(SPANISH_NEG_DIR)) < SPANISH_SAMPLES // 2:
    os.makedirs(SPANISH_NEG_DIR, exist_ok=True)
    print(f'⏳ Descargando Common Voice ES ({SPANISH_SAMPLES} muestras)...')
    print('   Nota: requiere aceptar términos de Mozilla en HuggingFace')

    try:
        ds = hf_datasets.load_dataset(
            'mozilla-foundation/common_voice_13_0',
            'es',
            split='train',
            streaming=True,
            trust_remote_code=True
        )

        count = 0
        for row in tqdm(ds, desc='Common Voice ES', total=SPANISH_SAMPLES):
            if count >= SPANISH_SAMPLES:
                break
            try:
                audio = row['audio']
                arr = np.array(audio['array'], dtype=np.float32)
                sr = audio['sampling_rate']

                # Resamplear a 16kHz si es necesario
                if sr != 16000:
                    from scipy.signal import resample_poly
                    arr = resample_poly(arr, 16000, sr).astype(np.float32)

                # Normalizar y guardar
                max_val = np.max(np.abs(arr))
                if max_val > 0:
                    arr = arr / max_val * 0.85  # Headroom

                out_path = f'{SPANISH_NEG_DIR}/es_{count:05d}.wav'
                scipy.io.wavfile.write(out_path, 16000, (arr * 32767).astype(np.int16))
                count += 1
            except Exception:
                pass  # Saltar filas con error

        print(f'✅ {count} muestras de habla española guardadas.')

    except Exception as e:
        print(f'⚠️  Error descargando Common Voice: {e}')
        print('   Alternativa: descarga manualmente de https://commonvoice.mozilla.org/es/datasets')
        print('   y extrae los .mp3 en ./negative_datasets/spanish_speech/')
else:
    n = len(os.listdir(SPANISH_NEG_DIR))
    print(f'✅ Negativos en español ya existen ({n} archivos).')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'mozilla-foundation/common_voice_13_0' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


⏳ Descargando Common Voice ES (4000 muestras)...
   Nota: requiere aceptar términos de Mozilla en HuggingFace


Repo card metadata block was not found. Setting CardData to empty.


⚠️  Error descargando Common Voice: The directory at hf://datasets/mozilla-foundation/common_voice_13_0@ff2bbb54dcdb597100fe534a1b911ff9103f9e22 doesn't contain any data files
   Alternativa: descarga manualmente de https://commonvoice.mozilla.org/es/datasets
   y extrae los .mp3 en ./negative_datasets/spanish_speech/


In [ ]:
"""---
## 📉 Descarga de Negativos (Falsos Positivos)
"""

import os
import soundfile as sf
from datasets import load_dataset
from tqdm import tqdm

NEG_DIR = "negative_datasets/spanish_speech"
os.makedirs(NEG_DIR, exist_ok=True)

SAMPLES_NEGATIVOS = 4000

print("⏳ Descargando audios negativos en español (Meta VoxPopuli - Formato Parquet)...")
try:
    # VoxPopuli está en formato nativo, inmune a los bloqueos de scripts de HuggingFace
    dataset = load_dataset("facebook/voxpopuli", "es", split="train", streaming=True)
    
    saved = 0
    with tqdm(total=SAMPLES_NEGATIVOS, desc="Descargando") as pbar:
        for item in dataset:
            if saved >= SAMPLES_NEGATIVOS:
                break
                
            audio_data = item['audio']['array']
            sr = item['audio']['sampling_rate']
            
            # Guardar el audio raw
            out_path = os.path.join(NEG_DIR, f"voxpopuli_es_{saved:04d}.wav")
            sf.write(out_path, audio_data, sr)
            
            saved += 1
            pbar.update(1)
            
    print(f"\n✅ ¡Descarga completada! {saved} audios guardados en '{NEG_DIR}/'")

except Exception as e:
    print(f"\n❌ Error durante la descarga: {e}")

## 🌊 Celda 7 — Descargar audios de fondo para augmentación
Ruidos de fondo reales hacen que el modelo sea más robusto en entornos ruidosos.

In [ ]:
import datasets as hf_datasets
import scipy.io.wavfile
import numpy as np
import os
from pathlib import Path
from tqdm import tqdm

# ── Respuestas de impulso (Room Impulse Responses) ────────────────────────────
RIR_DIR = './mit_rirs'
if not os.path.exists(RIR_DIR):
    print('⏳ Descargando MIT Room Impulse Responses...')
    os.makedirs(RIR_DIR, exist_ok=True)
    rir_dataset = hf_datasets.load_dataset(
        'davidscripka/MIT_environmental_impulse_responses',
        split='train', streaming=True
    )
    for row in tqdm(rir_dataset, desc='Procesando RIR'):
        name = row['audio']['path'].split('/')[-1]
        scipy.io.wavfile.write(
            os.path.join(RIR_DIR, name), 16000,
            (row['audio']['array'] * 32767).astype(np.int16)
        )
    print('✅ RIRs descargadas.')
else:
    print('✅ MIT RIRs ya existen.')

# ── AudioSet (ruidos y ambiente) ──────────────────────────────────────────────
AUDIOSET_16K_DIR = './audioset_16k'
if not os.path.exists(AUDIOSET_16K_DIR):
    AUDIOSET_RAW_DIR = './audioset'
    if not os.path.exists(AUDIOSET_RAW_DIR):
        print('⏳ Descargando AudioSet...')
        os.makedirs(AUDIOSET_RAW_DIR, exist_ok=True)
        link = 'https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar'
        os.system(f'wget -q --show-progress -O {AUDIOSET_RAW_DIR}/bal_train09.tar {link}')
        os.system(f'cd {AUDIOSET_RAW_DIR} && tar -xf bal_train09.tar')

    print('⚙️  Convirtiendo AudioSet a 16kHz...')
    os.makedirs(AUDIOSET_16K_DIR, exist_ok=True)
    flac_files = list(Path(f'{AUDIOSET_RAW_DIR}/audio').glob('**/*.flac'))
    ds = hf_datasets.Dataset.from_dict({'audio': [str(f) for f in flac_files]})
    ds = ds.cast_column('audio', hf_datasets.Audio(sampling_rate=16000))
    for row in tqdm(ds, desc='AudioSet → 16kHz'):
        name = row['audio']['path'].split('/')[-1].replace('.flac', '.wav')
        scipy.io.wavfile.write(
            os.path.join(AUDIOSET_16K_DIR, name), 16000,
            (row['audio']['array'] * 32767).astype(np.int16)
        )
    print('✅ AudioSet 16kHz listo.')
else:
    print('✅ AudioSet 16kHz ya existe.')

# ── Free Music Archive (música de fondo) ──────────────────────────────────────
FMA_16K_DIR = './fma_16k'
if not os.path.exists(FMA_16K_DIR):
    FMA_RAW_DIR = './fma'
    if not os.path.exists(FMA_RAW_DIR):
        print('⏳ Descargando Free Music Archive (FMA xsmall)...')
        os.makedirs(FMA_RAW_DIR, exist_ok=True)
        link = 'https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip'
        os.system(f'wget -q --show-progress -O {FMA_RAW_DIR}/fma_xs.zip {link}')
        os.system(f'cd {FMA_RAW_DIR} && unzip -q fma_xs.zip')

    print('⚙️  Convirtiendo FMA a 16kHz...')
    os.makedirs(FMA_16K_DIR, exist_ok=True)
    mp3_files = list(Path(f'{FMA_RAW_DIR}/fma_small').glob('**/*.mp3'))
    ds = hf_datasets.Dataset.from_dict({'audio': [str(f) for f in mp3_files]})
    ds = ds.cast_column('audio', hf_datasets.Audio(sampling_rate=16000))
    for row in tqdm(ds, desc='FMA → 16kHz'):
        name = row['audio']['path'].split('/')[-1].replace('.mp3', '.wav')
        scipy.io.wavfile.write(
            os.path.join(FMA_16K_DIR, name), 16000,
            (row['audio']['array'] * 32767).astype(np.int16)
        )
    print('✅ FMA 16kHz listo.')
else:
    print('✅ FMA 16kHz ya existe.')

print('\n✅ Todos los audios de fondo están listos.')

## 🔬 Celda 8 — Configurar augmentaciones y clips

La augmentación simula condiciones del mundo real **además** de la simulación INMP441 ya aplicada en la generación.

**Probabilidades ajustadas para ESP32-S3 + INMP441:**
- `SevenBandParametricEQ` reduce a 0.10 porque ya simulamos la respuesta del micrófono
- `AddColorNoise` sube a 0.30 para simular ruido eléctrico del bus I2S
- `RIR` se mantiene alto — la reverberación es el factor más crítico de robustez

In [ ]:
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

# ── Cargar clips generados por Kokoro ─────────────────────────────────────────
clips = Clips(
    input_directory=OUTPUT_DIR,
    file_pattern='*.wav',
    max_clip_duration_s=None,
    remove_silence=False,
    random_split_seed=42,
    split_count=0.1,   # 10% para validación/test
)
import glob
n = len(glob.glob(f'{OUTPUT_DIR}/*.wav'))
print(f'📂 Clips cargados: {n} archivos en {OUTPUT_DIR}/')

# ── Definir augmentaciones ────────────────────────────────────────────────────
# Ajustados para INMP441 + ESP32-S3 en condiciones reales (0.5–3 metros)
# La simulación INMP441 ya se aplicó en la generación, aquí modelamos
# el entorno acústico (sala, ruido de fondo, distancia)
augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.10,    # Reducido: INMP441 ya simulado
        'TanhDistortion':        0.10,    # Saturación ligera (micrófono sobrecargado)
        'PitchShift':            0.20,    # Variación de tono adicional (acento, edad)
        'BandStopFilter':        0.10,    # Simula interferencias de WiFi/BT del ESP32
        'AddColorNoise':         0.30,    # Subido: ruido eléctrico bus I2S + MEMS
        'AddBackgroundNoise':    0.85,    # Crítico — siempre mezclar con ambiente
        'Gain':                  1.00,    # Siempre normalizar volumen
        'RIR':                   0.65,    # Reverberación de sala (muy importante)
    },
    impulse_paths=['mit_rirs'],
    background_paths=['fma_16k', 'audioset_16k'],
    background_min_snr_db=-5,    # Puede ser más ruidoso que la voz en peor caso
    background_max_snr_db=20,    # SNR amplio para cubrir desde silencio hasta TV
    min_jitter_s=0.195,
    max_jitter_s=0.205,
)

print('✅ Augmentaciones configuradas.')

## 🎧 Celda 9 — Verificar augmentación con simulación INMP441

In [ ]:
from IPython.display import Audio, display
from microwakeword.audio.audio_utils import save_clip

random_clip = clips.get_random_clip()
augmented   = augmenter.augment_clip(random_clip)
save_clip(augmented, 'augmented_clip.wav')

print('🎧 Clip augmentado (con ruido de fondo, reverb y simulación INMP441):')
display(Audio('augmented_clip.wav', autoplay=True))

## 🗺️ Celda 10 — Generar espectrogramas para training/validation/testing
Convierte los clips de audio en espectrogramas (la representación que usa el modelo).  
Esta celda puede tardar varios minutos dependiendo del número de muestras.

In [ ]:
import os, shutil
from mmap_ninja.ragged import RaggedMmap

FEATURES_DIR = 'generated_augmented_features'
os.makedirs(FEATURES_DIR, exist_ok=True)

SPLITS = {
    'training':   {'split_name': 'train',      'repetition': 2,  'slide_frames': 10},
    'validation': {'split_name': 'validation', 'repetition': 1,  'slide_frames': 10},
    'testing':    {'split_name': 'test',       'repetition': 1,  'slide_frames': 1},
}

for split, cfg in SPLITS.items():
    out_dir  = os.path.join(FEATURES_DIR, split)
    mmap_dir = os.path.join(out_dir, 'wakeword_mmap')

    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)
        print(f'🗑️  {split}: limpiado.')
    os.makedirs(out_dir, exist_ok=True)

    print(f'⏳ Generando espectrogramas para: {split}...')
    spectrograms = SpectrogramGeneration(
        clips=clips,
        augmenter=augmenter,
        slide_frames=cfg['slide_frames'],
        step_ms=10,
    )

    RaggedMmap.from_generator(
        out_dir=mmap_dir,
        sample_generator=spectrograms.spectrogram_generator(
            split=cfg['split_name'],
            repeat=cfg['repetition']
        ),
        batch_size=100,
        verbose=True,
    )
    print(f'  ✅ {split} listo.')

print('\n✅ Todos los espectrogramas generados.')

## ⚙️ Celda 11 — Generar configuración de entrenamiento (YAML)

### Estrategia de entrenamiento en 2 fases para minimizar falsos positivos:

**Fase 1** — Aprendizaje general (20,000 pasos, `neg_weight=10`):  
El modelo aprende a reconocer la wake word y a ignorar ruido en general.

**Fase 2** — Fine-tuning anti-FP (5,000 pasos, `neg_weight=30`):  
Con LR bajo y penalización agresiva, refina los límites de decisión  
para rechazar agresivamente fonemas similares en español.

El objetivo `target_minimization: 0.5` es más estricto que el default (1.0),  
forzando al trainer a priorizar modelos con menos de 0.5 FP/hora.

In [ ]:
import yaml

# ══════════════════════════════════════════════════════════════════════════════
#  ⚙️  Parámetros de entrenamiento — 2 fases anti-FP
# ══════════════════════════════════════════════════════════════════════════════
#
# Fase 1: Aprendizaje general
# Fase 2: Fine-tuning con penalización agresiva de falsos positivos
#
TRAINING_STEPS        = [20000, 5000]    # Dos fases
POSITIVE_CLASS_WEIGHT = [1,     1]       # Peso de la wake word
NEGATIVE_CLASS_WEIGHT = [10,    30]      # Fase 2: muy agresivo contra FP
LEARNING_RATE         = [0.001, 0.0001]  # LR bajo en fine-tuning
# ══════════════════════════════════════════════════════════════════════════════

config = {
    'window_step_ms': 10,
    'clip_duration_ms': 1000,
    'average_window_duration_ms': 100,
    'detection_threshold': 0.5,
    'suppression_ms': 500,
    'minimum_count': 3,
    'batch_size': 128,
    'training_input_shape': [48, 40],
    'eval_step_interval': 500,
    'save_step_interval': 500,

    # Métricas de optimización — priorizar reducción de FP
    'minimization_metric': 'ambient_false_positives_per_hour',
    'maximization_metric': 'average_viable_recall',
    'target_minimization': 0.5,   # Objetivo: < 0.5 FP/hora (más estricto que default 1.0)
    'primary_metric': 'accuracy',

    'train_dir': 'trained_models/wakeword',

    # SpecAugment — regularización en espectrograma
    'time_mask_max_size': [5],
    'time_mask_count': [2],
    'freq_mask_max_size': [5],
    'freq_mask_count': [2],

    'features': [
        # ── Positivos (wake word generada con Kokoro + variantes fonéticas) ────
        {
            'features_dir': 'generated_augmented_features',
            'sampling_weight': 2.0,
            'penalty_weight': 1.0,
            'truth': True,
            'truncation_strategy': 'truncate_start',
            'type': 'mmap',
        },

        # ── Negativos: habla en ESPAÑOL (el más importante — reduce FP en español) ──
        {
            'features_dir': 'negative_datasets/spanish_speech',
            'sampling_weight': 15.0,   # El peso más alto de todos
            'penalty_weight': 2.5,     # Penalización extra — FP en español son los peores
            'truth': False,
            'truncation_strategy': 'random',
            'type': 'mmap',
        },

        # ── Negativos: speech general (en inglés del dataset original) ─────────
        {
            'features_dir': 'negative_datasets/speech',
            'sampling_weight': 8.0,
            'penalty_weight': 1.0,
            'truth': False,
            'truncation_strategy': 'random',
            'type': 'mmap',
        },

        # ── Negativos: conversaciones en grupo ───────────────────────────────
        {
            'features_dir': 'negative_datasets/dinner_party',
            'sampling_weight': 8.0,
            'penalty_weight': 1.0,
            'truth': False,
            'truncation_strategy': 'random',
            'type': 'mmap',
        },

        # ── Negativos: silencio / no-speech ──────────────────────────────────
        {
            'features_dir': 'negative_datasets/no_speech',
            'sampling_weight': 4.0,
            'penalty_weight': 1.0,
            'truth': False,
            'truncation_strategy': 'random',
            'type': 'mmap',
        },

        # ── Solo evaluación (no se usa en training) ───────────────────────────
        {
            'features_dir': 'negative_datasets/dinner_party_eval',
            'sampling_weight': 0.0,
            'penalty_weight': 1.0,
            'truth': False,
            'truncation_strategy': 'split',
            'type': 'mmap',
        },
    ],

    'training_steps':        TRAINING_STEPS,
    'positive_class_weight': POSITIVE_CLASS_WEIGHT,
    'negative_class_weight': NEGATIVE_CLASS_WEIGHT,
    'learning_rates':        LEARNING_RATE,
}

with open('training_parameters.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print('✅ training_parameters.yaml guardado.')
print(f'   Fase 1: {TRAINING_STEPS[0]:,} pasos, neg_weight={NEGATIVE_CLASS_WEIGHT[0]}')
print(f'   Fase 2: {TRAINING_STEPS[1]:,} pasos, neg_weight={NEGATIVE_CLASS_WEIGHT[1]} (anti-FP agresivo)')
print(f'   Objetivo: < {config["target_minimization"]} falsos positivos/hora')

## 🏋️ Celda 12 — Entrenamiento

Entrena el modelo `mixednet` optimizado para ESP32-S3.  
La GPU se utilizará automáticamente si está disponible.

**Arquitectura:** MixedNet con convoluciones mixtas — diseñado para ser pequeño (~50K MACs)  
y correr en microcontroladores. Las 2 fases de entrenamiento se ejecutan automáticamente.

Al final verás métricas como:
- `frr` = False Rejection Rate (la wake word se ignora) — debe ser < 0.05
- `faph` = False Accepts Per Hour (activa sin que digas la palabra) — objetivo < 0.5

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'🖥️  Dispositivos disponibles: GPU={len(gpus)}, CPU disponible')
if gpus:
    print(f'   GPU: {gpus[0].name}')
print(f'   TensorFlow: {tf.__version__}')

In [ ]:
import subprocess, os, sys

env = os.environ.copy()
env['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'
env['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
env['TF_CUDA_MALLOC_ASYNC_SUPPORTED_PREEMPTIVE_FREE_FRACTION'] = '0.5'
env['PYTHONIOENCODING'] = 'utf-8'
env['LANG'] = 'en_US.UTF-8'

cmd = [
    sys.executable, '-m', 'microwakeword.model_train_eval',
    '--training_config=training_parameters.yaml',
    '--train', '1',
    '--restore_checkpoint', '0',
    '--test_tf_nonstreaming', '0',
    '--test_tflite_nonstreaming', '0',
    '--test_tflite_nonstreaming_quantized', '0',
    '--test_tflite_streaming', '0',
    '--test_tflite_streaming_quantized', '1',
    '--use_weights', 'best_weights',
    'mixednet',
    '--pointwise_filters', '64,64,64,64',
    '--repeat_in_block', '1,1,1,1',
    '--mixconv_kernel_sizes', '[5], [7,11], [9,15], [23]',
    '--residual_connection', '0,0,0,0',
    '--first_conv_filters', '32',
    '--first_conv_kernel_size', '5',
    '--stride', '2',
]

process = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, encoding='utf-8')
for line in process.stdout:
    print(line, end='')
process.wait()

if process.returncode == 0:
    print('\n✅ Entrenamiento completado exitosamente.')
else:
    print(f'\n❌ El entrenamiento terminó con código {process.returncode}')

## 📊 Celda 13 — Ver métricas con TensorBoard (opcional)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir trained_models/wakeword

## 📦 Celda 14 — Exportar modelo para ESP32-S3

El archivo `.tflite` es el modelo final cuantizado en INT8 que corre directamente en la ESP32-S3.

### Parámetros ESPHome optimizados para baja tasa de FP:
- `probability_cutoff: 0.92` — más estricto que el default 0.5
- `sliding_window_size: 10` — requiere 100ms de activación continua (elimina FP cortos)

**Ajusta `probability_cutoff` según las métricas:**
- Si `faph > 1.0` → sube a 0.94–0.96
- Si `frr > 0.10` (se pierde la wake word) → baja a 0.88–0.90
- El rango óptimo suele estar entre 0.88 y 0.95

In [ ]:
import os, shutil

TFLITE_SOURCE = 'trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite'
TFLITE_DEST   = f'{TARGET_WORD.replace(" ", "_")}_esp32s3.tflite'

if os.path.exists(TFLITE_SOURCE):
    shutil.copy(TFLITE_SOURCE, TFLITE_DEST)
    size_kb = os.path.getsize(TFLITE_DEST) / 1024
    print(f'✅ Modelo exportado: {TFLITE_DEST}')
    print(f'   Tamaño: {size_kb:.1f} KB')
    print(f'''
📋 Ejemplo de manifiesto ESPHome (model_manifest.json):
{{
  "type": "micro_wake_word_model",
  "wake_word": "{TARGET_WORD}",
  "author": "Tu nombre",
  "website": "",
  "version": "1",
  "micro": {{
    "model": "stream_state_internal_quant.tflite",
    "probability_cutoff": 0.92,
    "sliding_window_size": 10,
    "tensor_arena_size": 40000
  }}
}}

💡 Si sigues teniendo FP: sube probability_cutoff a 0.94 o 0.96
💡 Si el modelo no detecta la wake word: baja probability_cutoff a 0.88
    ''')
else:
    print('❌ No se encontró el archivo .tflite. ¿El entrenamiento finalizó correctamente?')
    print(f'   Buscado en: {TFLITE_SOURCE}')

In [ ]:
# ── Descarga directa ───────────────────────────────────────────────────────────
# En Colab: from google.colab import files; files.download(TFLITE_DEST)
# En local: el archivo ya está en el directorio de trabajo

if os.path.exists(TFLITE_DEST):
    print(f'📥 El archivo está en: {os.path.abspath(TFLITE_DEST)}')
    
    # Si estás en Google Colab, descomenta:
    # from google.colab import files
    # files.download(TFLITE_DEST)